In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow_hub as hub
import os
import string

2026-03-21 10:10:27.616111: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 10:10:28.731558: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-21 10:10:32.910557: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [64]:
train_path="/mnt/d/Python/DeepLearningGPU/Project_Basic_Text_Classification/stack_overflow_16k/train"
test_path="/mnt/d/Python/DeepLearningGPU/Project_Basic_Text_Classification/stack_overflow_16k/test"

In [65]:
raw_train_data=tf.keras.utils.text_dataset_from_directory(train_path,seed=42,batch_size=32,validation_split=0.2,subset="training")
raw_val_data=tf.keras.utils.text_dataset_from_directory(train_path,seed=42,batch_size=32,validation_split=0.2,subset="validation")

Found 8000 files belonging to 4 classes.
Using 6400 files for training.
Found 8000 files belonging to 4 classes.
Using 1600 files for validation.


In [66]:
raw_test_data=tf.keras.utils.text_dataset_from_directory(test_path,batch_size=32)

Found 8000 files belonging to 4 classes.


In [67]:
def custom_standardizer(text):
    text=tf.strings.lower(text)
    text=tf.strings.regex_replace(text,'python|java|javascript|csharp'," ")
    return text


In [68]:
vectorized_layer=tf.keras.layers.TextVectorization(max_tokens=10000,
                                                   standardize=custom_standardizer,
                                                   output_mode="int",
                                                   output_sequence_length=250)

In [79]:
def vectorized_text(text,label):
    # text=tf.expand_dims(text,-1)
    return vectorized_layer(text),label

In [80]:
train_text=raw_train_data.map(lambda x,y:x)
vectorized_layer.adapt(train_text)

In [81]:
text_batch,label_batch=next(iter(raw_test_data))
print(text_batch[0])
print(raw_test_data.class_names[label_batch[0]])
print(vectorized_text(text_batch[0],label_batch[0]))

tf.Tensor(b'"is it bad practice in blank to modify input object in void method? in blank, assume you have a data object object with an attribute bar that you need to set with a value that is returned from a complex operation done in an external source. assume you have a method sendrequesttoexternalsource that send a request based on \'object\' to the external source and gets an object back holding (among other things) the needed value...which one of these ways to set the value is the better practice?..void main(myobject object) {.    bar = sendrequesttoexternalsource(object);.    object.setbar(bar);.}..string sendrequesttoexternalsource(myobject object) {..    // send request to external source.    object response = posttoexternalsource(object);..    //do some validation and logic based on response.    .....    //return only the attribute we are interested in.    return response.getbar();.}...or..void main(myobject object) {.    sendrequesttoexternalsourceandupdateobject(object);.}..vo

In [82]:
test_ds=raw_test_data.map(vectorized_text)
val_ds=raw_val_data.map(vectorized_text)
train_ds=raw_train_data.map(vectorized_text)

In [83]:
model=tf.keras.Sequential([
    tf.keras.layers.Embedding(10000,16),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(4,activation="softmax")
])
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_7      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [84]:
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=["accuracy"])

In [85]:
autotune=tf.data.AUTOTUNE
train_ds=train_ds.cache().prefetch(buffer_size=autotune)
test_ds=test_ds.cache().prefetch(buffer_size=autotune)
val_ds=val_ds.cache().prefetch(buffer_size=autotune)

In [86]:
callback=tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',
                                          patience=3,
                                          restore_best_weights=True)

In [87]:
model.fit(train_ds,validation_data=val_ds,epochs=40,callbacks=callback)

Epoch 1/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.3536 - loss: 1.3693 - val_accuracy: 0.3837 - val_loss: 1.3491
Epoch 2/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4411 - loss: 1.3157 - val_accuracy: 0.4462 - val_loss: 1.2818
Epoch 3/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5036 - loss: 1.2345 - val_accuracy: 0.5181 - val_loss: 1.1938
Epoch 4/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5686 - loss: 1.1467 - val_accuracy: 0.6275 - val_loss: 1.1044
Epoch 5/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6269 - loss: 1.0656 - val_accuracy: 0.6862 - val_loss: 1.0256
Epoch 6/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6655 - loss: 0.9892 - val_accuracy: 0.6844 - val_loss: 0.9589
Epoch 7/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6895 - loss: 0.9252 - val_accuracy: 0.7056 - val_loss: 0.9000
Epoch 8/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7095 - loss: 0.8671 - val_accuracy: 

In [78]:
model.evaluate(test_ds)

250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.7807 - loss: 0.5913


[0.5913223028182983, 0.7807499766349792]